# 第三章：机械手臂顺运动学 — 高级交互式仿真

> 台大林沛群教授《机器人学》第三章

| Part | 内容 |
|------|------|
| 1 | DH 四步分解动画 |
| 2 | RP 极坐标机械臂 |
| 3 | RRR 独立网页（Three.js） |
| 4 | PUMA 560（已并入 Part 9，保留 FK helper） |
| 5 | Kinematic Mapping：关节空间 → 笛卡尔空间 |
| 6 | 工作空间 3D 可视化 |
| 7 | 批量导出静态图 |
| 8 | Meshcat 3D 可视化 |
| 9 | Three.js 独立网页（综合 viewer） |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D
import warnings
from pathlib import Path
from ipywidgets import FloatSlider, interactive_output, VBox, HBox
from IPython.display import display
from matplotlib import font_manager as fm

warnings.filterwarnings('ignore')
%matplotlib inline

# CJK Font
def _pick_font():
    want = ['Microsoft YaHei', 'Microsoft YaHei UI', 'SimHei', 'NSimSun',
            'Microsoft JhengHei', 'DejaVu Sans']
    installed = {f.name for f in fm.fontManager.ttflist}
    for n in want:
        if n in installed:
            return n
    return 'DejaVu Sans'

CJK = _pick_font()
plt.rcParams.update({'font.family': CJK, 'font.sans-serif': [CJK, 'DejaVu Sans'],
                     'axes.unicode_minus': False, 'figure.dpi': 110})

# Modified DH 变换矩阵
# 顺序 (a, alpha_deg, d, theta_deg) 与林教授 DH 表一致
# ^{i-1}T_i = Rx(α) · Tx(a) · Rz(θ) · Tz(d)
def mdh(a, alpha_deg, d, theta_deg):
    α = np.radians(alpha_deg)
    θ = np.radians(theta_deg)
    ca, sa = np.cos(α), np.sin(α)
    ct, st = np.cos(θ), np.sin(θ)
    return np.array([
        [ ct,    -st,      0,    a    ],
        [ st*ca,  ct*ca,  -sa, -d*sa ],
        [ st*sa,  ct*sa,   ca,  d*ca ],
        [ 0,      0,       0,   1    ]
    ])

def fk(dh_rows):
    """连乘 Modified DH 链，返回各坐标系 T 列表（含基座 I）"""
    T, frames = np.eye(4), [np.eye(4)]
    for row in dh_rows:
        T = T @ mdh(*row)
        frames.append(T.copy())
    return frames

# 3D 绘图工具
RC = ['#e74c3c', '#27ae60', '#2980b9']  # R G B

def draw_frame(ax, T, scale=0.12, labels=('x','y','z'), alpha=1.0, lw=2):
    o, R = T[:3, 3], T[:3, :3]
    for i, (c, lbl) in enumerate(zip(RC, labels)):
        v = R[:, i] * scale
        ax.quiver(*o, *v, color=c, alpha=alpha, arrow_length_ratio=0.22, linewidth=lw)
        if lbl:
            ax.text(*(o + R[:, i]*scale*1.4), lbl, color=c, fontsize=8, fontweight='bold')

def draw_arm(ax, frames, lc='#4a90d9', jc='#c0392b', lw=4):
    pts = np.array([f[:3, 3] for f in frames])
    for i in range(len(pts)-1):
        ax.plot(*zip(pts[i], pts[i+1]), color=lc, lw=lw, solid_capstyle='round')
    ax.scatter(*pts[:-1].T, color=jc, s=55, zorder=5, depthshade=False)
    ax.scatter(*pts[-1],    color='#e74c3c', s=160, zorder=6, marker='*', depthshade=False)
    return pts

def setup_3d(ax, lim=0.8, title='', elev=20, azim=40):
    ax.set_xlim(-lim,lim);x ax.set_ylim(-lim,lim); ax.set_zlim(-lim,lim)
    ax.set_xlabel('X',fontsize=9); ax.set_ylabel('Y',fontsize=9); ax.set_zlabel('Z',fontsize=9)
    ax.set_box_aspect([1,1,1]); ax.view_init(elev=elev, azim=azim)
    if title: ax.set_title(title, fontsize=10, fontweight='bold')
    ax.tick_params(labelsize=7)
    ax.xaxis.pane.fill = ax.yaxis.pane.fill = ax.zaxis.pane.fill = False

def _show_fig(fig):
    """在 notebook 中只显示一次，避免重复富输出。"""
    plt.tight_layout()
    display(fig)
    plt.close(fig)

print(f'✓ 加载完成  |  字体: {CJK}')


✓ 加载完成  |  字体: Microsoft YaHei


---
## Part 1：DH 四步分解动画

一次 Modified DH 变换分四步完成：

$${}^{i-1}_iT = \underbrace{R_x(\alpha_{i-1})}_{\text{Step 1}} \cdot \underbrace{T_x(a_{i-1})}_{\text{Step 2}} \cdot \underbrace{R_z(\theta_i)}_{\text{Step 3}} \cdot \underbrace{T_z(d_i)}_{\text{Step 4}}$$

拖动滑块，观察每一步如何改变坐标系。

In [2]:
_s_a     = FloatSlider(value=0.6, min=0.0, max=1.2, step=0.1,  description='a')
_s_alpha = FloatSlider(value=30,  min=-90,  max=90,  step=15,   description='α (deg)')
_s_theta = FloatSlider(value=45,  min=-180, max=180, step=15,   description='θ (deg)')
_s_d     = FloatSlider(value=0.3, min=-0.8, max=0.8, step=0.1,  description='d')

def show_dh_steps(a, alpha, theta, d):
    plt.close('all')
    # 四步中间状态
    T0 = np.eye(4)
    T1 = T0 @ mdh(0, alpha, 0, 0)    # step1: Rx(α)
    T2 = T1 @ mdh(a, 0,     0, 0)    # step2: Tx(a)
    T3 = T2 @ mdh(0, 0,     0, theta)# step3: Rz(θ)
    T4 = T3 @ mdh(0, 0,     d, 0)    # step4: Tz(d)

    steps = [(T0, 'Step0: 基座 {i-1}'),
             (T1, f'Step1: Rx(α={alpha:.0f}°)'),
             (T2, f'Step2: Tx(a={a:.1f})'),
             (T3, f'Step3: Rz(θ={theta:.0f}°)'),
             (T4, f'Step4: Tz(d={d:.1f})  →  坐标系 {{i}}')]

    fig = plt.figure(figsize=(16, 4))
    for k, (T, title) in enumerate(steps):
        ax = fig.add_subplot(1, 5, k+1, projection='3d')
        # 始终画出基座（灰色，浅）
        draw_frame(ax, np.eye(4), scale=0.4, alpha=0.2,
                   labels=(None,None,None), lw=1)
        draw_frame(ax, T, scale=0.55, labels=('x','y','z'), lw=2)
        # 在 step>=2 时，画一条从前一步到当前步的平移线
        if k >= 2:
            prev_T = steps[k-1][0]
            ax.plot(*zip(prev_T[:3,3], T[:3,3]), '--', color='gray', alpha=0.6, lw=1.5)
        setup_3d(ax, lim=1.0, title=title, elev=18, azim=35)

    plt.suptitle('Modified DH 四步分解  (每步坐标系用 RGB = xyz 表示)',
                 fontsize=12, fontweight='bold', y=1.01)
    _show_fig(fig)

_out1 = interactive_output(show_dh_steps,
    {'a': _s_a, 'alpha': _s_alpha, 'theta': _s_theta, 'd': _s_d})
display(VBox([HBox([_s_a, _s_alpha]), HBox([_s_theta, _s_d]), _out1]))


---
## Part 2：RP 极坐标机械臂

- **Joint 1（R）**：转动关节，关节变量 $\theta$
- **Joint 2（P）**：移动关节，关节变量 $r$（沿连杆方向伸缩）

$$x = r\cos\theta, \quad y = r\sin\theta$$

关节空间是个矩形 $(\theta, r)$，笛卡尔空间里是**扇环**。

In [3]:
_rp_theta = FloatSlider(value=45, min=-180, max=180, step=5, description='θ (deg)')
_rp_r     = FloatSlider(value=0.8, min=0.2, max=1.5, step=0.05, description='r')

def show_rp(theta, r):
    plt.close('all')
    t = np.radians(theta)
    ee = np.array([r * np.cos(t), r * np.sin(t)])

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 5))

    # 机械臂结构图
    ax1.set_aspect('equal')
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(-1.8, 1.8)
    ax1.set_ylim(-1.8, 1.8)
    ax1.set_title('RP 机械臂当前姿态', fontsize=11, fontweight='bold')
    # 画出当前臂
    ax1.annotate('', xy=ee, xytext=(0,0),
                 arrowprops=dict(arrowstyle='->', color='#2980b9', lw=3))
    ax1.scatter(*ee, color='crimson', s=120, zorder=5)
    ax1.scatter(0, 0, color='#333', s=80, zorder=5)
    ax1.text(ee[0]+0.05, ee[1]+0.05, f'EE=({ee[0]:.2f},{ee[1]:.2f})', fontsize=9)
    # 画角度弧
    arc_t = np.linspace(0, t, 60)
    arc_r = 0.25
    ax1.plot(arc_r*np.cos(arc_t), arc_r*np.sin(arc_t), 'k-', lw=1.5)
    ax1.text(0.28*np.cos(t/2), 0.28*np.sin(t/2), f'θ={theta:.0f}°', fontsize=9)
    ax1.set_xlabel('X')
    ax1.set_ylabel('Y')

    # θ 变化轨迹（r 固定）
    ax2.set_aspect('equal')
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(-1.8, 1.8)
    ax2.set_ylim(-1.8, 1.8)
    ax2.set_title('θ 变化（r 固定）→ 圆弧', fontsize=11)
    ts = np.linspace(-np.pi, np.pi, 200)
    ax2.plot(r*np.cos(ts), r*np.sin(ts), color='#8e44ad', lw=2, alpha=0.7, label=f'r={r:.2f}')
    ax2.scatter(*ee, color='crimson', s=80, zorder=5)
    ax2.set_xlabel('X')
    ax2.set_ylabel('Y')
    ax2.legend()

    # r 变化轨迹（θ 固定）
    ax3.set_aspect('equal'); ax3.grid(True, alpha=0.3)
    ax3.set_xlim(-1.8, 1.8); ax3.set_ylim(-1.8, 1.8)
    ax3.set_title('r 变化（θ 固定）→ 射线', fontsize=11)
    rs = np.linspace(0.2, 1.5, 100)
    ax3.plot(rs*np.cos(t), rs*np.sin(t), color='#e67e22', lw=2, alpha=0.8, label=f'θ={theta:.0f}°')
    ax3.scatter(*ee, color='crimson', s=80, zorder=5)
    ax3.set_xlabel('X'); ax3.set_ylabel('Y'); ax3.legend()

    plt.suptitle(f'RP 极坐标机械臂  |  θ={theta:.0f}°   r={r:.2f}',
                 fontsize=12, fontweight='bold')
    _show_fig(fig)

_out2 = interactive_output(show_rp, {'theta': _rp_theta, 'r': _rp_r})
display(VBox([HBox([_rp_theta, _rp_r]), _out2]))


---
## Part 3：RRR 机械臂（Three.js 独立网页）

这一节不再使用 notebook 内联 Matplotlib / MuJoCo 渲染。  
原因有两个：

1. 课程里真正想看的，是 RRR 链条本身与末端姿态，而不是某个特定 notebook 图形后端。
2. Matplotlib 在 Jupyter 里本来就会自动显示 figure；再配合显式 `show()` 或 widget 输出，容易出现重复出图。

因此这里改成和最终综合 viewer 同一套技术栈：

- **Three.js**
- **OrbitControls**
- **独立 HTML 文件**

你运行下面这格后，会生成 `rrr_fk_viewer.html`。  
它只保留 RRR，适合讲解最基础的三连杆顺运动学。


In [4]:
import textwrap
from IPython.display import HTML as IPHTML, display

L1, L2, L3 = 0.50, 0.40, 0.30

def rrr_fk(t1, t2, t3):
    dh = [(0, 0, 0, t1), (L1, 0, 0, t2), (L2, 0, 0, t3), (L3, 0, 0, 0)]
    return fk(dh)

RRR_HTML = textwrap.dedent(r"""<!DOCTYPE html><html lang="zh-CN"><head><meta charset="UTF-8">
<title>RRR Forward Kinematics Viewer</title>
<style>
:root{--bg:#0d1117;--panel:#161b22;--border:#21262d;--text:#c9d1d9;--muted:#8b949e;--acc:#58a6ff}
*{margin:0;padding:0;box-sizing:border-box}
body{height:100vh;display:flex;background:var(--bg);color:var(--text);font:13px/1.5 'Segoe UI',system-ui,sans-serif;overflow:hidden}
#panel{width:290px;min-width:290px;background:var(--panel);border-right:1px solid var(--border);display:flex;flex-direction:column}
.ph{padding:14px 16px;border-bottom:1px solid var(--border)}
.ph h1{font-size:15px;font-weight:600;color:var(--acc)}
.ph p{font-size:11px;color:var(--muted);margin-top:2px}
#sliders{flex:1;overflow-y:auto;padding:8px 0}
.sr{padding:10px 16px}
.sh{display:flex;justify-content:space-between;margin-bottom:4px}
.jn{font-weight:500;font-size:13px}.jv{color:var(--acc);font-family:monospace;font-size:12px}
input[type=range]{width:100%;height:3px;accent-color:var(--acc);cursor:pointer}
#info{padding:12px 16px;background:rgba(0,0,0,.28);border-top:1px solid var(--border)}
.ig{display:grid;grid-template-columns:auto 1fr;gap:4px 10px}
.ik{color:var(--muted);font-size:12px}.iv{color:var(--acc);font-family:monospace;font-size:12px;text-align:right}
#cv{flex:1;position:relative}canvas{display:block}
</style></head><body>
<div id="panel">
  <div class="ph">
    <h1>RRR 顺运动学</h1>
    <p>Three.js 独立网页版本</p>
  </div>
  <div id="sliders"></div>
  <div id="info">
    <div class="ig">
      <span class="ik">x</span><span class="iv" id="vx">—</span>
      <span class="ik">y</span><span class="iv" id="vy">—</span>
      <span class="ik">z</span><span class="iv" id="vz">—</span>
      <span class="ik">φ</span><span class="iv" id="vp">—</span>
    </div>
  </div>
</div>
<div id="cv"></div>

<script type="importmap">
{"imports":{"three":"https://cdn.jsdelivr.net/npm/three@0.160.0/build/three.module.js","three/addons/":"https://cdn.jsdelivr.net/npm/three@0.160.0/examples/jsm/"}}
</script>
<script type="module">
import * as T from 'three';
import {OrbitControls} from 'three/addons/controls/OrbitControls.js';

const wrap=document.getElementById('cv');
const ren=new T.WebGLRenderer({antialias:true});
ren.setPixelRatio(devicePixelRatio);
ren.shadowMap.enabled=true; ren.shadowMap.type=T.PCFSoftShadowMap;
ren.toneMapping=T.ACESFilmicToneMapping; ren.toneMappingExposure=1.15;
wrap.appendChild(ren.domElement);

const scene=new T.Scene(); scene.background=new T.Color(0x0d1117);
scene.fog=new T.FogExp2(0x0d1117,.12);
const cam=new T.PerspectiveCamera(42,1,.01,60);
const ctrl=new OrbitControls(cam,ren.domElement);
ctrl.enableDamping=true; ctrl.dampingFactor=.06;
function resize(){const w=wrap.clientWidth,h=wrap.clientHeight; ren.setSize(w,h); cam.aspect=w/h; cam.updateProjectionMatrix();}
window.addEventListener('resize',resize); resize();

scene.add(new T.AmbientLight(0x8090b0,.9));
const sun=new T.DirectionalLight(0xffffff,3.5);
sun.position.set(3,4,3); sun.castShadow=true; sun.shadow.mapSize.set(2048,2048);
Object.assign(sun.shadow.camera,{left:-2,right:2,top:2,bottom:-2,near:.1,far:20});
scene.add(sun);
const fill=new T.DirectionalLight(0x8090ff,.6); fill.position.set(-2,2,-1); scene.add(fill);
const floor=new T.Mesh(new T.PlaneGeometry(8,8), new T.MeshStandardMaterial({color:0x1a1f2e,roughness:.9}));
floor.rotation.x=-Math.PI/2; floor.receiveShadow=true; scene.add(floor);
scene.add(new T.GridHelper(4,20,0x2a2d40,0x1e2130));

const MAT={
  base:new T.MeshStandardMaterial({color:0x2c3e50,roughness:.4,metalness:.75}),
  joint:new T.MeshStandardMaterial({color:0xe8eaf6,roughness:.15,metalness:.9}),
  ee:new T.MeshStandardMaterial({color:0xff3030,roughness:.1,metalness:.9,emissive:0x440000}),
  lk:[0x1976d2,0x388e3c,0xf57c00,0x00acc1].map(c=>new T.MeshStandardMaterial({color:c,roughness:.3,metalness:.6}))
};

function mdh(a,aDeg,d,tDeg){
  const al=aDeg*Math.PI/180, th=tDeg*Math.PI/180;
  const [ca,sa,ct,st]=[Math.cos(al),Math.sin(al),Math.cos(th),Math.sin(th)];
  const M=new T.Matrix4();
  M.set(ct,-st,0,a, st*ca,ct*ca,-sa,-d*sa, st*sa,ct*sa,ca,d*ca, 0,0,0,1);
  return M;
}
function chainFK(rows){
  let T0=new T.Matrix4(); const fs=[T0.clone()];
  for(const r of rows){T0=T0.clone().multiply(mdh(...r)); fs.push(T0.clone());}
  return fs;
}
function pos3(M){const e=M.elements; return new T.Vector3(e[12],e[13],e[14]);}
function toThree(v){return new T.Vector3(v.x,v.z+0.12,-v.y);}

const linkRadii=[.055,.055,.045,.035];
const root=new T.Group(); scene.add(root);
const joints=[], links=[];
const Y1=new T.Vector3(0,1,0);

function mkCap(r,l){return new T.CapsuleGeometry(r,Math.max(.001,l),10,20);}
function setLink(mesh,p1,p2,r){
  const dir=new T.Vector3().subVectors(p2,p1), L=dir.length();
  mesh.position.copy(p1.clone().add(p2).multiplyScalar(.5));
  mesh.quaternion.setFromUnitVectors(Y1,dir.clone().normalize());
  mesh.geometry.dispose(); mesh.geometry=mkCap(r,Math.max(.001,L-2*r));
}
function buildScene(){
  const bc=new T.Mesh(new T.CylinderGeometry(.10,.12,.12,32),MAT.base);
  bc.position.y=.06; bc.castShadow=bc.receiveShadow=true; root.add(bc);
  const br=new T.Mesh(new T.CylinderGeometry(.09,.09,.015,32),MAT.joint);
  br.position.y=.127; root.add(br);
  for(let i=0;i<4;i++){
    const lm=new T.Mesh(mkCap(linkRadii[i],.1),MAT.lk[i].clone());
    lm.castShadow=true; root.add(lm); links.push(lm);
  }
  for(let i=0;i<3;i++){
    const jm=new T.Mesh(new T.SphereGeometry(linkRadii[Math.min(i+1,3)]*1.5,32,32),MAT.joint.clone());
    jm.castShadow=true; root.add(jm); joints.push(jm);
  }
  const ee=new T.Mesh(new T.SphereGeometry(.042,32,32),MAT.ee.clone());
  ee.castShadow=true; root.add(ee); joints.push(ee);
}
buildScene();

const cfg=[{l:'θ₁',mn:-180,mx:180,v:30},{l:'θ₂',mn:-180,mx:180,v:60},{l:'θ₃',mn:-180,mx:180,v:-45}];
function buildSliders(){
  const c=document.getElementById('sliders');
  cfg.forEach((j,i)=>{
    const d=document.createElement('div'); d.className='sr';
    d.innerHTML=`<div class="sh"><span class="jn">${j.l}</span><span class="jv" id="jv${i}">${j.v.toFixed(0)}°</span></div><input type="range" id="js${i}" min="${j.mn}" max="${j.mx}" step="1" value="${j.v}">`;
    d.querySelector('input').addEventListener('input', ()=>{
      document.getElementById('jv'+i).textContent=parseFloat(d.querySelector('input').value).toFixed(0)+'°';
      update();
    });
    c.appendChild(d);
  });
}
buildSliders();

function update(){
  const a=[0,1,2].map(i=>parseFloat(document.getElementById('js'+i).value));
  const frames=chainFK([[0,0,0,a[0]],[.5,0,0,a[1]],[.4,0,0,a[2]],[.3,0,0,0]]);
  const pts=frames.map(f=>toThree(pos3(f)));
  for(let i=0;i<4;i++)setLink(links[i],pts[i],pts[i+1],linkRadii[i]);
  for(let i=0;i<3;i++)joints[i].position.copy(pts[i+1]);
  joints[3].position.copy(pts[4]);
  const ee=pos3(frames[4]);
  document.getElementById('vx').textContent=ee.x.toFixed(4);
  document.getElementById('vy').textContent=ee.y.toFixed(4);
  document.getElementById('vz').textContent=ee.z.toFixed(4);
  document.getElementById('vp').textContent=(a[0]+a[1]+a[2]).toFixed(1)+'°';
  const box=new T.Box3().setFromPoints(pts), c=box.getCenter(new T.Vector3()), s=box.getSize(new T.Vector3());
  const r=Math.max(s.length()*.55,.7);
  ctrl.target.copy(c);
  cam.position.copy(c.clone().add(new T.Vector3(r*1.9,r*1.35,r*1.9)));
  cam.lookAt(c);
}
update();
(function anim(){requestAnimationFrame(anim); ctrl.update(); ren.render(scene,cam);})();
</script></body></html>""").strip()

def _resolve_notebook_dir():
    cwd = Path.cwd()
    for base in (cwd, cwd / 'Robotics_NTU'):
        if (base / 'Chapter03_visualization.ipynb').exists():
            return base
    return cwd

NOTEBOOK_DIR = _resolve_notebook_dir()
rrr_html = NOTEBOOK_DIR / 'rrr_fk_viewer.html'
rrr_html.write_text(RRR_HTML, encoding='utf-8')

display(IPHTML(
    f'<div style="padding:10px 12px;border:1px solid #d0d7de;border-radius:8px;">'
    f'<b>已生成 RRR 独立网页：</b> '
    f'<a href="{rrr_html.as_uri()}" target="_blank">{rrr_html.name}</a><br>'
    f'<span style="color:#57606a;">同 Part 9 一样使用 Three.js + OrbitControls，不再依赖 MuJoCo / notebook 内联后端。</span>'
    f'</div>'
))
print(f'文件位置: {rrr_html}')


文件位置: d:\Code\Learning\EI\EI-learning-notes\Robotics_NTU\rrr_fk_viewer.html


---
## Part 4：PUMA 560 六自由度顺运动学（已并入 Part 9）

旧的 Matplotlib 交互版已停用，避免与 Part 9 的 Three.js viewer 重复。  
这里仅保留 PUMA 560 的 FK 定义，供 Part 7 的静态导图与公式复用。


In [5]:
PUMA560_STD_DH = [
    (0,      90,  26.45 * 0.0254, 0),  # pedestal / shoulder axis height
    (0.4318, 0,   0,              0),
    (0.0203,-90,  0.15005,        0),
    (0,      90,  0.4318,         0),
    (0,     -90,  0,              0),
    (0,       0,  0,              0),
]

def sdh(a, alpha_deg, d, theta_deg):
    """Standard DH: Rz(theta) · Tz(d) · Tx(a) · Rx(alpha)"""
    α = np.radians(alpha_deg)
    θ = np.radians(theta_deg)
    ca, sa = np.cos(α), np.sin(α)
    ct, st = np.cos(θ), np.sin(θ)
    return np.array([
        [ct, -st*ca,  st*sa, a*ct],
        [st,  ct*ca, -ct*sa, a*st],
        [0,      sa,     ca,    d],
        [0,       0,      0,    1],
    ])

def fk_standard(dh_rows):
    T, frames = np.eye(4), [np.eye(4)]
    for row in dh_rows:
        T = T @ sdh(*row)
        frames.append(T.copy())
    return frames

def puma_fk(t1, t2, t3, t4, t5, t6):
    thetas = [t1, t2, t3, t4, t5, t6]
    rows = [(PUMA560_STD_DH[i][0], PUMA560_STD_DH[i][1],
             PUMA560_STD_DH[i][2], thetas[i]) for i in range(6)]
    return fk_standard(rows)


---
## Part 5：Kinematic Mapping（关节空间 → 笛卡尔空间）

林教授强调：**关节空间里的直线，映射到笛卡尔空间不一定是直线。**

下面同时展示四条轨迹：线性插值 / 圆弧 / 摆动 / 螺旋形，观察两个空间的对应关系。

In [6]:
def fk_planar_ee(t1_deg, t2_deg, l1=1.2, l2=0.9):
    t1, t2 = np.radians(t1_deg), np.radians(t2_deg)
    x = l1*np.cos(t1) + l2*np.cos(t1+t2)
    y = l1*np.sin(t1) + l2*np.sin(t1+t2)
    return x, y

_km_t1s = FloatSlider(value=10,  min=-160, max=160, step=5, description='θ₁ start')
_km_t2s = FloatSlider(value=20,  min=-160, max=160, step=5, description='θ₂ start')
_km_t1e = FloatSlider(value=80,  min=-160, max=160, step=5, description='θ₁ end')
_km_t2e = FloatSlider(value=-40, min=-160, max=160, step=5, description='θ₂ end')

def show_kinematic_map(t1s, t2s, t1e, t2e):
    plt.close('all')
    N = 120
    ts = np.linspace(0, 1, N)

    # 四种关节空间轨迹
    traj_js = {
        '线性插值': (t1s + (t1e-t1s)*ts,
                    t2s + (t2e-t2s)*ts),
        '圆弧（关节空间）': (t1s + (t1e-t1s)*ts,
                            t2s + (t2e-t2s)*np.sin(ts*np.pi/2)),
        '摆动': (t1s + (t1e-t1s)*ts,
                  t2s + (t2e-t2s)*ts + 30*np.sin(ts*4*np.pi)),
        '慢起快落': (t1s + (t1e-t1s)*ts**2,
                     t2s + (t2e-t2s)*ts**2),
    }
    colors = ['#2980b9','#e74c3c','#27ae60','#8e44ad']

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # 关节空间
    ax1.set_title('关节空间 (θ₁, θ₂)', fontsize=12, fontweight='bold')
    ax1.set_xlabel('θ₁ (°)'); ax1.set_ylabel('θ₂ (°)')
    ax1.grid(True, alpha=0.3)
    for (name, (q1, q2)), c in zip(traj_js.items(), colors):
        ax1.plot(q1, q2, color=c, lw=2.5, label=name)
        ax1.scatter([q1[0],q1[-1]], [q2[0],q2[-1]],
                    color=['#2ecc71','#e74c3c'], s=60, zorder=5)
    ax1.legend(fontsize=9, loc='best')

    # 笛卡尔空间
    ax2.set_title('笛卡尔空间（末端轨迹）', fontsize=12, fontweight='bold')
    ax2.set_xlabel('X'); ax2.set_ylabel('Y')
    ax2.set_aspect('equal'); ax2.grid(True, alpha=0.3)
    # 画工作空间边界（参考）
    l1, l2 = 1.2, 0.9
    for R, sty in [(l1+l2,'--'), (abs(l1-l2),':')]:
        tc = np.linspace(0, 2*np.pi, 200)
        ax2.plot(R*np.cos(tc), R*np.sin(tc), sty, color='gray', alpha=0.4, lw=1)
    for (name, (q1, q2)), c in zip(traj_js.items(), colors):
        xs, ys = zip(*[fk_planar_ee(a, b, l1, l2) for a,b in zip(q1,q2)])
        ax2.plot(xs, ys, color=c, lw=2.5, label=name)
        ax2.scatter([xs[0],xs[-1]], [ys[0],ys[-1]],
                    color=['#2ecc71','#e74c3c'], s=60, zorder=5)
    ax2.legend(fontsize=9, loc='best')

    plt.suptitle('Kinematic Mapping：相同起终点，四种关节空间路径 → 四条不同末端轨迹',
                 fontsize=12, fontweight='bold')
    _show_fig(fig)

_out5 = interactive_output(show_kinematic_map,
    {'t1s':_km_t1s,'t2s':_km_t2s,'t1e':_km_t1e,'t2e':_km_t2e})
display(VBox([HBox([_km_t1s,_km_t2s]), HBox([_km_t1e,_km_t2e]), _out5]))


---
## Part 6：工作空间 3D 可视化

工作空间 = 末端执行器**所有可达位置**的集合。

改变各关节范围，观察工作空间形状如何变化。

In [ ]:
_ws_q1r = FloatSlider(value=120, min=10, max=180, step=10, description='±q1 范围')
_ws_q2r = FloatSlider(value=150, min=10, max=180, step=10, description='±q2 范围')
_ws_q3r = FloatSlider(value=150, min=10, max=180, step=10, description='±q3 范围')
_ws_n   = FloatSlider(value=40,  min=15, max=70,  step=5,  description='采样密度')

def show_workspace(q1r, q2r, q3r, n):
    plt.close('all')
    n = int(n)
    l1, l2, l3 = 0.5, 0.4, 0.3
    q1s = np.linspace(-q1r, q1r, n)
    q2s = np.linspace(-q2r, q2r, n)
    q3s = np.linspace(-q3r, q3r, n)

    # 均匀采样（降低计算量，用步长跳采）
    pts, phis = [], []
    for q1 in q1s[::2]:
        for q2 in q2s[::2]:
            for q3 in q3s[::2]:
                rows = [(0,0,0,q1),(l1,0,0,q2),(l2,0,0,q3),(l3,0,0,0)]
                ee = fk(rows)[-1][:3, 3]
                pts.append(ee)
                phis.append((q1+q2+q3) % 360)
    pts = np.array(pts)
    phis = np.array(phis)

    fig = plt.figure(figsize=(14, 6))

    # 3D 工作空间（按末端姿态角着色）
    ax1 = fig.add_subplot(121, projection='3d')
    sc = ax1.scatter(pts[:,0], pts[:,1], pts[:,2],
                     c=phis, cmap='hsv', s=3, alpha=0.25)
    plt.colorbar(sc, ax=ax1, label='末端姿态角 φ (°)', shrink=0.7)
    ax1.set_title('3D 工作空间（颜色=φ角）', fontsize=11, fontweight='bold')
    ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')
    lim = l1+l2+l3+0.05
    ax1.set_xlim(-lim,lim); ax1.set_ylim(-lim,lim); ax1.set_zlim(-lim,lim)
    ax1.set_box_aspect([1,1,1])

    # XY 俯视截面
    ax2 = fig.add_subplot(122)
    ax2.scatter(pts[:,0], pts[:,1], c=phis, cmap='hsv', s=3, alpha=0.2)
    ax2.set_aspect('equal'); ax2.grid(True, alpha=0.3)
    ax2.set_title('XY 平面工作空间截面', fontsize=11, fontweight='bold')
    ax2.set_xlabel('X'); ax2.set_ylabel('Y')
    for R, sty in [(l1+l2+l3,'--'), (abs(l1-l2-l3),':')]:
        tc = np.linspace(0, 2*np.pi, 300)
        ax2.plot(R*np.cos(tc), R*np.sin(tc), sty, color='black', alpha=0.5, lw=1.5)

    plt.suptitle(f'2R+R 工作空间  |  q1=±{q1r:.0f}°  q2=±{q2r:.0f}°  q3=±{q3r:.0f}°',
                 fontsize=12, fontweight='bold')
    _show_fig(fig)

_out6 = interactive_output(show_workspace,
    {'q1r':_ws_q1r,'q2r':_ws_q2r,'q3r':_ws_q3r,'n':_ws_n})
display(VBox([HBox([_ws_q1r, _ws_q2r]), HBox([_ws_q3r, _ws_n]), _out6]))


---
## Part 7：批量导出静态图
---


In [8]:
def _out_dir():
    cwd = Path.cwd()
    for base in (cwd, cwd/'Robotics_NTU'):
        if (base/'Chapter03_visualization.ipynb').exists():
            d = base/'images'; d.mkdir(exist_ok=True); return d
    d = cwd/'images'; d.mkdir(exist_ok=True); return d

OUTPUT_DIR = _out_dir()
print(f'导出目录: {OUTPUT_DIR}')

def _save(fig, name):
    p = OUTPUT_DIR / name
    fig.savefig(p, dpi=150, bbox_inches='tight'); plt.close(fig)
    return p

def export_all():
    l1, l2, l3 = 0.5, 0.4, 0.3
    paths = []

    # 1. DH 四步
    fig = plt.figure(figsize=(16,4))
    steps = [np.eye(4)]
    for fn in [lambda: mdh(0,30,0,0), lambda: mdh(0.6,0,0,0),
                lambda: mdh(0,0,0,45), lambda: mdh(0,0,0.3,0)]:
        steps.append(steps[-1] @ fn())
    titles = ['Step0','Step1: Rx(30°)','Step2: Tx(0.6)','Step3: Rz(45°)','Step4: Tz(0.3)']
    for k,(T,t) in enumerate(zip(steps,titles)):
        ax=fig.add_subplot(1,5,k+1,projection='3d')
        draw_frame(ax,T,scale=0.55,labels=('x','y','z'),lw=2)
        setup_3d(ax,lim=1.0,title=t)
    paths.append(_save(fig,'ch03_dh_steps.png'))

    # 2. RRR 静态图
    frames = rrr_fk(30, 60, -45)
    fig = plt.figure(figsize=(12,5))
    ax1 = fig.add_subplot(121, projection='3d')
    draw_arm(ax1, frames, lw=5)
    setup_3d(ax1, lim=l1+l2+l3+0.05, title='RRR 透视图', elev=24, azim=42)
    ax2 = fig.add_subplot(122, projection='3d')
    draw_arm(ax2, frames, lw=5)
    setup_3d(ax2, lim=l1+l2+l3+0.05, title='RRR 俯视图', elev=88, azim=90)
    paths.append(_save(fig,'ch03_rrr_threejs_reference.png'))

    # 3. PUMA 560
    frames = puma_fk(0,90,-90,0,90,0)
    fig=plt.figure(figsize=(7,6))
    ax=fig.add_subplot(111,projection='3d')
    draw_arm(ax,frames,lw=5)
    draw_frame(ax,frames[-1],scale=0.08,labels=('xe','ye','ze'))
    setup_3d(ax,lim=1.6,title='PUMA 560 · θ=[0,90,-90,0,90,0]°',elev=22,azim=48)
    paths.append(_save(fig,'ch03_puma560.png'))

    # 4. Kinematic Mapping
    N=120; ts=np.linspace(0,1,N)
    t1s,t2s,t1e,t2e=10,20,80,-40
    trajs={'线性': (t1s+(t1e-t1s)*ts, t2s+(t2e-t2s)*ts),
           '摆动': (t1s+(t1e-t1s)*ts, t2s+(t2e-t2s)*ts+30*np.sin(ts*4*np.pi))}
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(12,5))
    for (nm,(q1,q2)),c in zip(trajs.items(),['#2980b9','#e74c3c']):
        ax1.plot(q1,q2,color=c,lw=2,label=nm)
        xs,ys=zip(*[fk_planar_ee(a,b) for a,b in zip(q1,q2)])
        ax2.plot(xs,ys,color=c,lw=2,label=nm)
    ax1.legend(); ax1.grid(True,alpha=0.3); ax1.set_title('关节空间')
    ax2.legend(); ax2.grid(True,alpha=0.3); ax2.set_aspect('equal'); ax2.set_title('笛卡尔空间')
    paths.append(_save(fig,'ch03_kinematic_map.png'))

    # 5. Workspace
    pts=[]
    for q1 in np.linspace(-120,120,30):
        for q2 in np.linspace(-150,150,30):
            for q3 in np.linspace(-150,150,30):
                pts.append(fk([(0,0,0,q1),(l1,0,0,q2),(l2,0,0,q3),(l3,0,0,0)])[-1][:3,3])
    pts=np.array(pts)
    fig=plt.figure(figsize=(7,6))
    ax=fig.add_subplot(111,projection='3d')
    ax.scatter(pts[:,0],pts[:,1],pts[:,2],c=pts[:,2],cmap='plasma',s=2,alpha=0.2)
    ax.set_box_aspect([1,1,1]); ax.set_title('3D 工作空间')
    paths.append(_save(fig,'ch03_workspace3d.png'))

    print('导出完成：')
    for p in paths: print(f'  {p}')
    return paths

# 取消注释运行：
# export_all()


导出目录: d:\Code\Learning\EI\EI-learning-notes\Robotics_NTU\images


---
## Part 8：方案一 — Meshcat 3D 可视化（浏览器 WebGL）

`meshcat` 在本地启动一个 WebSocket 服务器，在浏览器里渲染 Three.js 场景。  
拖动 ipywidgets 滑块 → Python 实时推送变换到浏览器。
---


In [9]:
import meshcat
import meshcat.geometry as mc_g
import meshcat.transformations as mc_tf
from ipywidgets import FloatSlider, VBox, HBox
from IPython.display import display

# 创建 Visualizer（自动开启本地服务器）
vis = meshcat.Visualizer()
print("浏览器访问：", vis.url())
display(vis.jupyter_cell())   # 也可内嵌在 notebook 里

# 坐标转换：机器人 Z-up → meshcat/Three.js Y-up
def r2m(p):
    """(xr, yr, zr) → (xr, zr, -yr)"""
    return np.array([p[0], p[2], -p[1]], dtype=float)

# 辅助：在两点之间放置圆柱
def _capsule(vis, name, p1, p2, radius, color):
    d = p2 - p1;  L = np.linalg.norm(d)
    if L < 1e-6: return
    y = np.array([0., 1., 0.])
    d_hat = d / L
    cross = np.cross(y, d_hat);  cn = np.linalg.norm(cross)
    if cn < 1e-6:
        R4 = np.eye(4) if d_hat[1] > 0 else np.diag([1., -1., -1., 1.])
    else:
        cross /= cn
        a = np.arccos(np.clip(np.dot(y, d_hat), -1, 1))
        K = np.array([[0,-cross[2],cross[1]],[cross[2],0,-cross[0]],[-cross[1],cross[0],0]])
        R3 = np.eye(3) + np.sin(a)*K + (1-np.cos(a))*K@K
        R4 = np.eye(4);  R4[:3,:3] = R3
    R4[:3, 3] = (p1 + p2) / 2
    vis[name].set_object(mc_g.Cylinder(L, radius),
                         mc_g.MeshPhongMaterial(color=color, reflectivity=0.5))
    vis[name].set_transform(R4)

def _sphere(vis, name, pos, r, color):
    vis[name].set_object(mc_g.Sphere(r), mc_g.MeshPhongMaterial(color=color))
    vis[name].set_transform(mc_tf.translation_matrix(pos))

# RRR 场景更新
_L1, _L2, _L3, _BH = 0.5, 0.4, 0.3, 0.15
LCOLORS = [0x1976d2, 0x388e3c, 0xf57c00]

def _update_mc(t1=30., t2=60., t3=-45.):
    t1r, t2r, t3r = np.radians([t1, t2, t3])
    p0 = np.array([0., 0., _BH])
    p1 = p0 + _L1 * np.array([np.cos(t1r),       np.sin(t1r),       0.])
    p2 = p1 + _L2 * np.array([np.cos(t1r+t2r),   np.sin(t1r+t2r),   0.])
    p3 = p2 + _L3 * np.array([np.cos(t1r+t2r+t3r), np.sin(t1r+t2r+t3r), 0.])
    pts = [r2m(p) for p in [p0, p1, p2, p3]]

    # 底座
    _capsule(vis, 'arm/base', r2m([0,0,0]), r2m([0,0,_BH]), 0.07, 0x2c3e50)
    # 连杆
    for i, (pa, pb, c) in enumerate(zip(pts, pts[1:], LCOLORS)):
        _capsule(vis, f'arm/link{i+1}', pa, pb, [0.048,0.040,0.032][i], c)
    # 关节球
    for i, (p, r) in enumerate(zip(pts, [0.068,0.058,0.048,0.040])):
        _sphere(vis, f'arm/joint{i}', p, r, 0xe8eaf6)
    # 末端执行器
    _sphere(vis, 'arm/ee', pts[-1], 0.045, 0xff3030)
    # 地板网格（只需画一次）
    vis['floor'].set_object(mc_g.Box([2, 0.002, 2]),
                            mc_g.MeshPhongMaterial(color=0x1a1f2e))

# 滑块 + observe 回调
_s1 = FloatSlider(value=30,  min=-180, max=180, step=5, description='θ₁ (°)', style={'description_width':'60px'})
_s2 = FloatSlider(value=60,  min=-180, max=180, step=5, description='θ₂ (°)', style={'description_width':'60px'})
_s3 = FloatSlider(value=-45, min=-180, max=180, step=5, description='θ₃ (°)', style={'description_width':'60px'})

def _on_change(_):
    _update_mc(_s1.value, _s2.value, _s3.value)

for _s in [_s1, _s2, _s3]:
    _s.observe(_on_change, names='value')

_update_mc()   # 初始渲染
display(HBox([_s1, _s2, _s3]))
print("✓ 拖动滑块 → 浏览器实时更新")


You can open the visualizer by visiting the following URL:
http://127.0.0.1:7000/static/
浏览器访问： http://127.0.0.1:7000/static/


✓ 拖动滑块 → 浏览器实时更新


---
## Part 9：方案三 — Three.js 独立网页（综合 viewer）

这是本 notebook 里最稳定、也最接近最终成品的一种呈现方式。  
它直接生成一个独立 HTML 文件，用浏览器打开即可，不依赖 notebook 的 Matplotlib 后端。

它主要适合看三类东西：

1. **RRR**  
   用来理解最基础的三连杆平面链与末端姿态角 `φ = θ₁ + θ₂ + θ₃`。

2. **PUMA 560**  
   前三轴主要决定 wrist center 的位置；后三轴主要改变球腕与 tool tip 的方向。  
   因此 viewer 里专门把末端 tool tip 画出来，这样 `θ₄/θ₅/θ₆` 的作用会更直观。

3. **RRRP / PRRR**  
   用来比较 prismatic joint 放在末端和放在基座时，整条运动链的视觉差异。

运行下一格后会生成 `robot_fk_viewer.html`。  
如果你只想讲 RRR 基础版，就用 Part 3 生成的 `rrr_fk_viewer.html`；  
如果你想讲多构型对比，就用这里的综合 viewer。
---


In [10]:
import textwrap
from IPython.display import HTML as IPHTML, display

HTML = textwrap.dedent(r"""<!DOCTYPE html><html lang="zh-CN"><head><meta charset="UTF-8">
<title>正运动学 — 台大机器人学</title>
<style>
:root{--bg:#0d1117;--panel:#161b22;--border:#21262d;--text:#c9d1d9;--muted:#8b949e;--acc:#58a6ff}
*{margin:0;padding:0;box-sizing:border-box}
body{height:100vh;display:flex;background:var(--bg);color:var(--text);
  font:13px/1.5 'Segoe UI',system-ui,sans-serif;overflow:hidden}
#panel{width:290px;min-width:290px;background:var(--panel);border-right:1px solid var(--border);
  display:flex;flex-direction:column;overflow:hidden}
.ph{padding:14px 16px;border-bottom:1px solid var(--border)}
.ph h1{font-size:15px;font-weight:600;color:var(--acc)}
.ph p{font-size:11px;color:var(--muted);margin-top:2px}
.tabs{display:flex;gap:6px;padding:10px 16px;border-bottom:1px solid var(--border);flex-wrap:wrap}
.tab{flex:1 1 calc(50% - 3px);padding:6px 4px;border:1px solid var(--border);background:transparent;color:var(--muted);
  border-radius:6px;cursor:pointer;font-size:11px;font-weight:500;transition:.15s}
.tab:hover{border-color:var(--acc);color:var(--text)}
.tab.on{background:var(--acc);border-color:var(--acc);color:#0d1117;font-weight:700}
#sliders{flex:1;overflow-y:auto;padding:4px 0}
.sr{padding:9px 16px}
.sh{display:flex;justify-content:space-between;margin-bottom:4px}
.jn{font-weight:500;font-size:13px}.jv{color:var(--acc);font-family:monospace;font-size:12px}
input[type=range]{width:100%;height:3px;accent-color:var(--acc);cursor:pointer}
.trow{display:flex;align-items:center;gap:8px;padding:8px 16px;border-top:1px solid var(--border)}
.sw{width:32px;height:18px;background:var(--border);border-radius:9px;cursor:pointer;position:relative;flex-shrink:0}
.sw.on{background:var(--acc)}
.sw::after{content:'';position:absolute;width:14px;height:14px;background:#fff;border-radius:50%;top:2px;left:2px;transition:left .15s}
.sw.on::after{left:16px}
#info{padding:12px 16px;background:rgba(0,0,0,.3);border-top:1px solid var(--border)}
.it{font-size:10px;color:var(--muted);text-transform:uppercase;letter-spacing:1px;margin-bottom:6px}
.ig{display:grid;grid-template-columns:auto 1fr;gap:3px 10px}
.ik{color:var(--muted);font-size:12px}.iv{color:var(--acc);font-family:monospace;font-size:12px;text-align:right}
#cv{flex:1;position:relative}canvas{display:block}
</style></head><body>
<div id="panel">
  <div class="ph"><h1>正运动学可视化</h1><p>台大林沛群教授《机器人学》第三章</p></div>
  <div class="tabs">
    <button class="tab on" id="tb-rrr"  onclick="setR('rrr')">RRR</button>
    <button class="tab"    id="tb-puma" onclick="setR('puma')">PUMA 560</button>
    <button class="tab"    id="tb-rrrp" onclick="setR('rrrp')">RRRP</button>
    <button class="tab"    id="tb-prrr" onclick="setR('prrr')">PRRR</button>
  </div>
  <div id="sliders"></div>
  <div class="trow"><div class="sw" id="sw-ax" onclick="togAx()"></div>
    <span style="font-size:12px;color:var(--muted)">显示坐标轴</span></div>
  <div id="info">
    <div class="it">末端执行器</div>
    <div class="ig">
      <span class="ik">x</span><span class="iv" id="vx">—</span>
      <span class="ik">y</span><span class="iv" id="vy">—</span>
      <span class="ik">z</span><span class="iv" id="vz">—</span>
      <span class="ik" id="lp">φ</span><span class="iv" id="vp">—</span>
    </div>
  </div>
</div>
<div id="cv"></div>

<script type="importmap">
{"imports":{"three":"https://cdn.jsdelivr.net/npm/three@0.160.0/build/three.module.js",
"three/addons/":"https://cdn.jsdelivr.net/npm/three@0.160.0/examples/jsm/"}}
</script>
<script type="module">
import * as T from 'three';
import {OrbitControls} from 'three/addons/controls/OrbitControls.js';

const wrap=document.getElementById('cv');
const ren=new T.WebGLRenderer({antialias:true});
ren.setPixelRatio(devicePixelRatio);
ren.shadowMap.enabled=true; ren.shadowMap.type=T.PCFSoftShadowMap;
ren.toneMapping=T.ACESFilmicToneMapping; ren.toneMappingExposure=1.15;
wrap.appendChild(ren.domElement);

const scene=new T.Scene(); scene.background=new T.Color(0x0d1117);
scene.fog=new T.FogExp2(0x0d1117,.12);
const cam=new T.PerspectiveCamera(42,1,.01,80);
cam.position.set(2.2,1.8,2.2);
const ctrl=new OrbitControls(cam,ren.domElement);
ctrl.enableDamping=true; ctrl.dampingFactor=.06;
ctrl.target.set(0,.4,0); ctrl.minDistance=.4; ctrl.maxDistance=18;

function resize(){
  const w=wrap.clientWidth,h=wrap.clientHeight;
  ren.setSize(w,h); cam.aspect=w/h; cam.updateProjectionMatrix();
}
window.addEventListener('resize',resize); resize();

scene.add(new T.AmbientLight(0x8090b0,.9));
const sun=new T.DirectionalLight(0xffffff,3.5);
sun.position.set(3,4,3); sun.castShadow=true;
sun.shadow.mapSize.set(2048,2048);
Object.assign(sun.shadow.camera,{left:-2,right:2,top:2,bottom:-2,near:.1,far:20});
scene.add(sun);
const fill=new T.DirectionalLight(0x8090ff,.6); fill.position.set(-2,2,-1); scene.add(fill);

const fm=new T.MeshStandardMaterial({color:0x1a1f2e,roughness:.9});
const fl=new T.Mesh(new T.PlaneGeometry(8,8),fm);
fl.rotation.x=-Math.PI/2; fl.receiveShadow=true; scene.add(fl);
scene.add(new T.GridHelper(4,20,0x2a2d40,0x1e2130));

const MAT={
  base:new T.MeshStandardMaterial({color:0x2c3e50,roughness:.4,metalness:.75}),
  joint:new T.MeshStandardMaterial({color:0xe8eaf6,roughness:.15,metalness:.9}),
  ee:new T.MeshStandardMaterial({color:0xff3030,roughness:.1,metalness:.9,emissive:0x440000}),
  lk:[0x1976d2,0x388e3c,0xf57c00,0x7b1fa2,0x00838f,0xc62828].map(
    c=>new T.MeshStandardMaterial({color:c,roughness:.3,metalness:.6}))
};

function mdh(a,aDeg,d,tDeg){
  const al=aDeg*Math.PI/180, th=tDeg*Math.PI/180;
  const [ca,sa,ct,st]=[Math.cos(al),Math.sin(al),Math.cos(th),Math.sin(th)];
  const M=new T.Matrix4();
  M.set(ct,-st,0,a, st*ca,ct*ca,-sa,-d*sa, st*sa,ct*sa,ca,d*ca, 0,0,0,1);
  return M;
}
function sdh(a,aDeg,d,tDeg){
  const al=aDeg*Math.PI/180, th=tDeg*Math.PI/180;
  const [ca,sa,ct,st]=[Math.cos(al),Math.sin(al),Math.cos(th),Math.sin(th)];
  const M=new T.Matrix4();
  M.set(ct,-st*ca,st*sa,a*ct, st,ct*ca,-ct*sa,a*st, 0,sa,ca,d, 0,0,0,1);
  return M;
}
function chainFK(rows){
  let T0=new T.Matrix4();
  const fs=[T0.clone()];
  for(const r of rows){T0=T0.clone().multiply(mdh(...r)); fs.push(T0.clone());}
  return fs;
}
function chainFKStandard(rows){
  let T0=new T.Matrix4();
  const fs=[T0.clone()];
  for(const r of rows){T0=T0.clone().multiply(sdh(...r)); fs.push(T0.clone());}
  return fs;
}
function pos3(M){const e=M.elements; return new T.Vector3(e[12],e[13],e[14]);}
function toThree(v){return new T.Vector3(v.x,v.z,-v.y);}
function rawPoints(frames,cfg){
  const pts=frames.map(f=>pos3(f));
  if(cfg.tipLen){
    const tip=pos3(frames[frames.length-1]);
    const axis=new T.Vector3().setFromMatrixColumn(frames[frames.length-1],0).normalize();
    pts.push(tip.clone().add(axis.multiplyScalar(cfg.tipLen)));
  }
  return pts;
}

const ROBOTS={
  rrr:{
    joints:[{l:'θ₁',mn:-180,mx:180,v:30},{l:'θ₂',mn:-180,mx:180,v:60},{l:'θ₃',mn:-180,mx:180,v:-45}],
    frames:a=>chainFK([[0,0,0,a[0]],[.5,0,0,a[1]],[.4,0,0,a[2]],[.3,0,0,0]]),
    linkRadii:[.055,.055,.045,.035], floorGap:.12, phiIndices:[0,1,2]
  },
  puma:{
    joints:[{l:'θ₁',mn:-180,mx:180,v:0},{l:'θ₂',mn:-135,mx:135,v:90},
      {l:'θ₃',mn:-135,mx:135,v:-90},{l:'θ₄',mn:-180,mx:180,v:0},
      {l:'θ₅',mn:-100,mx:100,v:0},{l:'θ₆',mn:-266,mx:266,v:0}],
    frames:a=>chainFKStandard([
      [0, 90, 26.45*0.0254, a[0]],
      [0.4318, 0, 0, a[1]],
      [0.0203, -90, 0.15005, a[2]],
      [0, 90, 0.4318, a[3]],
      [0, -90, 0, a[4]],
      [0, 0, 0, a[5]],
    ]),
    linkRadii:[.06,.055,.048,.04,.032,.024,.018], floorGap:.08, phiIndices:null, tipLen:.18
  },
  rrrp:{
    joints:[{l:'θ₁',mn:-180,mx:180,v:40},{l:'θ₂',mn:-180,mx:180,v:60},
      {l:'θ₃',mn:-180,mx:180,v:-30},{l:'d₄',mn:0,mx:.6,v:.2,step:.01,unit:'m'}],
    frames:a=>chainFK([[0,0,0,a[0]],[.45,0,0,a[1]],[.35,0,0,a[2]],[.25,0,0,0],[a[3],0,0,0]]),
    linkRadii:[.055,.055,.045,.03,.026], floorGap:.12, phiIndices:[0,1,2]
  },
  prrr:{
    joints:[{l:'d₁',mn:0,mx:.8,v:.3,step:.01,unit:'m'},{l:'θ₁',mn:-180,mx:180,v:40},
      {l:'θ₂',mn:-180,mx:180,v:60},{l:'θ₃',mn:-180,mx:180,v:-30}],
    frames:a=>chainFK([[0,0,a[0],0],[0,0,0,a[1]],[.45,0,0,a[2]],[.35,0,0,a[3]],[.25,0,0,0]]),
    linkRadii:[.05,.05,.045,.038,.03], floorGap:.12, phiIndices:[1,2,3]
  }
};

const root=new T.Group(); scene.add(root);
let objs={joints:[],links:[],axH:[],ee:null};
const Y1=new T.Vector3(0,1,0);

function mkCap(r,l){return new T.CapsuleGeometry(r,Math.max(.001,l),10,20);}
function rebuild(cfg){
  while(root.children.length)root.remove(root.children[0]);
  objs={joints:[],links:[],axH:[],ee:null};
  const n=cfg.joints.length, nLinks=cfg.linkRadii.length;
  const bc=new T.Mesh(new T.CylinderGeometry(.10,.12,.12,32),MAT.base);
  bc.position.y=.06; bc.castShadow=bc.receiveShadow=true; root.add(bc);
  const br=new T.Mesh(new T.CylinderGeometry(.09,.09,.015,32),MAT.joint);
  br.position.y=.127; root.add(br);
  for(let i=0;i<n;i++){
    const r=(cfg.linkRadii[Math.min(i+1,cfg.linkRadii.length-1)]||.025);
    const jm=new T.Mesh(new T.SphereGeometry(r*1.5,32,32),MAT.joint.clone());
    jm.castShadow=true; root.add(jm); objs.joints.push(jm);
    const ah=new T.AxesHelper(.16); ah.visible=false; root.add(ah); objs.axH.push(ah);
  }
  for(let i=0;i<nLinks;i++){
    const r=(cfg.linkRadii[i]||.025);
    const lm=new T.Mesh(mkCap(r,.1),MAT.lk[i%6].clone());
    lm.castShadow=true; root.add(lm); objs.links.push(lm);
  }
  const em=new T.Mesh(new T.SphereGeometry(.042,32,32),MAT.ee.clone());
  em.castShadow=true; root.add(em); objs.ee=em;
}

function setLink(mesh,p1,p2,r){
  const dir=new T.Vector3().subVectors(p2,p1), L=dir.length();
  if(L<.001){mesh.visible=false; return;}
  mesh.visible=true;
  mesh.position.copy(p1.clone().add(p2).multiplyScalar(.5));
  mesh.quaternion.setFromUnitVectors(Y1,dir.clone().normalize());
  mesh.geometry.dispose();
  mesh.geometry=mkCap(r,Math.max(.001,L-2*r));
}

function liftedPoints(rawPts,cfg){
  const pts=rawPts.map(p=>toThree(p));
  const minY=Math.min(...pts.map(p=>p.y));
  const lift=(cfg.floorGap??.08)-minY;
  return pts.map(p=>p.clone().add(new T.Vector3(0,lift,0)));
}

function fitCamera(points){
  const box=new T.Box3().setFromPoints(points);
  const c=box.getCenter(new T.Vector3());
  const s=box.getSize(new T.Vector3());
  const radius=Math.max(s.length()*.55,.7);
  ctrl.target.copy(c);
  cam.position.copy(c.clone().add(new T.Vector3(radius*1.9,radius*1.35,radius*1.9)));
  cam.lookAt(c);
  cam.far=Math.max(80,radius*30);
  cam.updateProjectionMatrix();
  ctrl.update();
}

let curRobot='rrr', showAx=false;

function updatePose(angles,refit=false){
  const cfg=ROBOTS[curRobot];
  const frames=cfg.frames(angles);
  const rawPts=rawPoints(frames,cfg);
  const pts=liftedPoints(rawPts,cfg);

  for(let i=0;i<objs.links.length;i++){
    setLink(objs.links[i],pts[i],pts[i+1],cfg.linkRadii[i]||.025);
  }

  for(let i=0;i<objs.joints.length;i++){
    const frameIdx=Math.min(i+1, pts.length-2);
    const p=pts[frameIdx];
    objs.joints[i].position.copy(p);
    objs.axH[i].position.copy(p); objs.axH[i].visible=showAx;
  }

  const ee=pts[pts.length-1];
  objs.ee.position.copy(ee);

  const er=rawPts[rawPts.length-1];
  document.getElementById('vx').textContent=er.x.toFixed(4);
  document.getElementById('vy').textContent=er.y.toFixed(4);
  document.getElementById('vz').textContent=er.z.toFixed(4);

  if(cfg.phiIndices){
    const phi=cfg.phiIndices.reduce((acc,idx)=>acc+(angles[idx]||0),0);
    document.getElementById('lp').textContent='φ=θ₁+θ₂+θ₃';
    document.getElementById('vp').textContent=phi.toFixed(1)+'°';
  }else{
    document.getElementById('lp').textContent='φ';
    document.getElementById('vp').textContent='—';
  }

  if(refit)fitCamera(pts);
}

function buildSliders(cfg){
  const c=document.getElementById('sliders'); c.innerHTML='';
  cfg.joints.forEach((j,i)=>{
    const step=j.step||1, unit=j.unit||'°';
    const d=document.createElement('div'); d.className='sr';
    d.innerHTML=`<div class="sh"><span class="jn">${j.l}</span>
      <span class="jv" id="jv${i}">${j.v.toFixed(unit==='m'?2:0)}${unit}</span></div>
      <input type="range" id="js${i}" min="${j.mn}" max="${j.mx}" step="${step}" value="${j.v}">`;
    d.querySelector('input').addEventListener('input',()=>{
      const a=ROBOTS[curRobot].joints.map((_,k)=>{
        const el=document.getElementById('js'+k);
        return el?parseFloat(el.value):0;
      });
      ROBOTS[curRobot].joints.forEach((joint,k)=>{
        const u=joint.unit||'°';
        const el=document.getElementById('jv'+k);
        if(el)el.textContent=a[k].toFixed(u==='m'?2:0)+u;
      });
      updatePose(a,false);
    });
    c.appendChild(d);
  });
}

window.setR=function(name){
  curRobot=name;
  document.querySelectorAll('.tab').forEach(t=>t.classList.remove('on'));
  document.getElementById('tb-'+name).classList.add('on');
  const cfg=ROBOTS[name];
  buildSliders(cfg);
  rebuild(cfg);
  updatePose(cfg.joints.map(j=>j.v),true);
};

window.togAx=function(){
  showAx=!showAx;
  document.getElementById('sw-ax').classList.toggle('on',showAx);
  objs.axH.forEach(a=>{a.visible=showAx;});
};

setR('rrr');
(function anim(){requestAnimationFrame(anim); ctrl.update(); ren.render(scene,cam);})();
</script></body></html>""").strip()

def _resolve_notebook_dir():
    cwd = Path.cwd()
    for base in (cwd, cwd / 'Robotics_NTU'):
        if (base / 'Chapter03_visualization.ipynb').exists():
            return base
    return cwd

NOTEBOOK_DIR = _resolve_notebook_dir()
out_html = NOTEBOOK_DIR / 'robot_fk_viewer.html'
out_html.write_text(HTML, encoding='utf-8')

display(IPHTML(
    f'<div style="padding:10px 12px;border:1px solid #d0d7de;border-radius:8px;">'
    f'<b>已生成综合 viewer：</b> '
    f'<a href="{out_html.as_uri()}" target="_blank">{out_html.name}</a><br>'
    f'<span style="color:#57606a;">包含 RRR / PUMA 560 / RRRP / PRRR，适合课程展示与讲解。</span>'
    f'</div>'
))
print(f'文件位置: {out_html}')


文件位置: d:\Code\Learning\EI\EI-learning-notes\Robotics_NTU\robot_fk_viewer.html
